In [ ]:
import json
from typing import Dict, List

file_path = r"..\data\crawler\weibo_trending\weibo_trending_data.json"
def read_data(file_path: str) -> List[Dict]:
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    return data

trendings = read_data(file_path)

len(trendings), trendings[:5]

(157437,
 [{'date': '2025-01-01', 'name': '赵露思发长文回应', 'type': '暂无', 'clicks': 14913080},
  {'date': '2025-01-01', 'name': '新年快乐', 'type': '其他', 'clicks': 10647970},
  {'date': '2025-01-01', 'name': '种地吧直播', 'type': '暂无', 'clicks': 3189152},
  {'date': '2025-01-01',
   'name': '总书记的这些话暖心鼓劲',
   'type': '社会',
   'clicks': 2512950},
  {'date': '2025-01-01', 'name': '银河酷娱致歉', 'type': '明星', 'clicks': 2425505}])

In [52]:
from collections import Counter

type_counter = Counter(item["type"].strip() for item in trendings
                       if len(item["type"]) < 10)

print("词条数最多的前20个类型：")
for type_name, count in type_counter.most_common(20):
    print(f"  {type_name}: {count:,} 条")

词条数最多的前20个类型：
  社会: 48,700 条
  暂无: 44,536 条
  明星: 13,936 条
  体育: 8,185 条
  明星-内地: 4,836 条
  时事: 4,680 条
  游戏: 4,240 条
  电视剧: 3,472 条
  财经: 3,219 条
  综艺: 2,599 条
  电视剧-国产剧: 1,701 条
  搞笑: 1,540 条
  综艺-内地综艺: 1,319 条
  电影: 1,298 条
  汽车: 1,282 条
  互联网: 865 条
  其他: 813 条
  音乐: 691 条
  科普: 675 条
  情感: 653 条


In [53]:
import random

selected_types = ["社会", "时事", "财经", "互联网", "科普", "情感"]

selected_trendings = [item for item in trendings if item["type"] in selected_types]

print(f"筛选后词条数: {len(selected_trendings)}")
print(f"占比: {len(selected_trendings) / len(trendings) * 100:.1f}%")
print("筛选词条示例：")

for item in random.sample(selected_trendings, 5):
    print(f" {item['date']} - {item['name']} - [{item['type']}] - {item['clicks']:,} 次")

筛选后词条数: 58792
占比: 37.3%
筛选词条示例：
 2025-01-19 - 多款手机价格集体降至6000元以内 - [社会] - 435,739 次
 2025-03-15 - 我国各区域外贸开年成绩单出炉 - [社会] - 908,466 次
 2025-12-13 - 日军南京杀人竞赛有人杀了106人 - [时事] - 225,995 次
 2025-04-24 - 妻子开车躲家暴致丈夫身亡被判11年 - [社会] - 689,231 次
 2025-03-11 - 95后女教师勇救轻生男孩 - [社会] - 235,235 次


In [54]:
import numpy as np

# 提取所有点击量
clicks = [item["clicks"] for item in selected_trendings]

ratio = 99

threshold = np.percentile(clicks, ratio)

print(f"总词条数: {len(selected_trendings)}")
print(f"点击量{ratio}%分位数: {threshold:,.0f}")
print(f"点击量范围: {min(clicks):,.0f} - {max(clicks):,.0f}")
# print(f"平均点击量: {np.mean(clicks):,.0f}")
# print(f"中位数点击量: {np.median(clicks):,.0f}")

top_1_percent_trendings = [item for item in selected_trendings 
                            if item["clicks"] >= threshold]

# 按点击量降序排序
top_1_percent_trendings.sort(key=lambda x: x["clicks"], reverse=True)

print(f"\n筛选后词条数: {len(top_1_percent_trendings)}")
print(f"占比: {len(top_1_percent_trendings) / len(trendings) * 100:.1f}%")

总词条数: 58792
点击量99%分位数: 1,991,206
点击量范围: 0 - 25,245,679

筛选后词条数: 588
占比: 0.4%


In [55]:
print("热搜词条示例：")
for item in random.sample(top_1_percent_trendings, 5):
    print(f"{item['date']} - {item['name']} - [{item['type']}] - {item['clicks']:,} 次")


热搜词条示例：
2025-04-10 - 美国被加关税后特朗普呼吁冷静 - [时事] - 7,605,554 次
2025-08-15 - 董某莹成绩单造假 - [社会] - 9,610,327 次
2025-09-23 - 桦加沙登陆地点确认 - [社会] - 2,052,809 次
2025-07-27 - 数智新时代电商新价值 - [社会] - 2,254,021 次
2025-01-07 - 王星弟弟已与其通视频电话 - [社会] - 10,831,049 次


In [56]:
# 查看筛选后的数据统计
print("=" * 60)
print("筛选后数据统计")
print("=" * 60)
print(f"总词条数: {len(top_1_percent_trendings):,}")
print(f"点击量阈值: {threshold:,.0f} 次")
print(f"最高点击量: {top_1_percent_trendings[0]['clicks']:,} 次")
print(f"最低点击量: {top_1_percent_trendings[-1]['clicks']:,} 次")
print(f"平均点击量: {np.mean([item['clicks'] for item in top_1_percent_trendings]):,.0f} 次")

# 按类型统计
from collections import Counter
type_dist = Counter(item['type'] for item in top_1_percent_trendings)
print(f"\n类型分布:")
for type_name, count in type_dist.most_common():
    percentage = count / len(top_1_percent_trendings) * 100
    print(f"  {type_name}: {count:,} 条 ({percentage:.1f}%)")

筛选后数据统计
总词条数: 588
点击量阈值: 1,991,206 次
最高点击量: 25,245,679 次
最低点击量: 1,991,504 次
平均点击量: 4,394,284 次

类型分布:
  社会: 503 条 (85.5%)
  时事: 39 条 (6.6%)
  财经: 31 条 (5.3%)
  互联网: 12 条 (2.0%)
  情感: 2 条 (0.3%)
  科普: 1 条 (0.2%)


In [58]:
with open("top_1_percent_trendings_detail.txt", 'w', encoding='utf-8') as f:
    for item in top_1_percent_trendings:
        f.write(f"{item['date']} - {item['name']} - [{item['type']}] - {item['clicks']:,}\n")

trending_names = set([item['name'] for item in top_1_percent_trendings])

with open("top_1_percent_trendings.txt", 'w', encoding='utf-8') as f:
    for name in trending_names:
        f.write(f"{name}\n")